### Manual Tracker (Object Detection + Object Tracking)

In [27]:
import cv2

video_path = '../samples/highway-2.mp4'

def resize_frame(frame):
  return cv2.resize(frame, (0, 0), fx=0.3, fy=0.3)

def select_roi(frame):
  frame = resize_frame(frame)
  roi = cv2.selectROI('roi', frame, showCrosshair=False)
  cv2.destroyWindow('roi')
  return roi

def extract_roi(frame, roi):
  roi_x, roi_y, roi_w, roi_h = roi
  return frame[roi_y:roi_y+roi_h, roi_x:roi_x+roi_w]

Renderizando o vídeo com o OpenCV

In [ ]:
cap = cv2.VideoCapture(video_path)

while True:
  ret, frame = cap.read()

  if not ret:
    break

  frame = resize_frame(frame)
  cv2.imshow('frame', frame)

  key = cv2.waitKey(30)

  if key == ord('q'):
    break

cap.release()
cv2.destroyAllWindows()


Utilizando estratégia de Object Detection para câmera estática

In [4]:
# Extração de objetos que se movem
object_detector = cv2.createBackgroundSubtractorMOG2()

In [ ]:
cap = cv2.VideoCapture(video_path)

while True:
  ret, frame = cap.read()

  if not ret:
    break

  frame = resize_frame(frame)
  mask = object_detector.apply(frame)
  contours, _ = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

  for contour in contours:
    # Calcular a área do contorno e remover pequenos contornos
    area = cv2.contourArea(contour)

    if area > 100:
      cv2.drawContours(frame, [contour], -1, (0, 255, 0), 2)

  cv2.imshow('frame', frame)
  cv2.imshow('mask', mask)

  key = cv2.waitKey(1)

  if key == ord('q'):
    break

cap.release()
cv2.destroyAllWindows()

Implementando ROI para limitar a área de interesse

In [6]:
# Aprimorando a detecção de objetos
object_detector = cv2.createBackgroundSubtractorMOG2(history=100, varThreshold=40)

In [ ]:
cap = cv2.VideoCapture(video_path)

_, first_frame = cap.read()

# Extrair região de interesse (ROI)
selected_roi = select_roi(first_frame)

while True:
  ret, frame = cap.read()

  if not ret:
    break

  frame = resize_frame(frame)

  # Extrair o ROI do frame atual
  roi = extract_roi(frame, selected_roi)

  # Detectar objetos em movimento dentro do ROI
  mask = object_detector.apply(roi)

  # Remover sombras
  _, mask = cv2.threshold(mask, 254, 255, cv2.THRESH_BINARY)

  contours, _ = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

  for contour in contours:
    # Calcular a área do contorno e remover pequenos contornos
    area = cv2.contourArea(contour)

    if area > 100:
      # cv2.drawContours(cur_roi, [contour], -1, (0, 255, 0), 2)
      x, y, w, h = cv2.boundingRect(contour)
      cv2.rectangle(roi, (x, y), (x+w, y+h), (0, 255, 0), 2)

  cv2.imshow('frame', frame)
  cv2.imshow('mask', mask)

  key = cv2.waitKey(30)

  if key == ord('q'):
    break

cap.release()
cv2.destroyAllWindows()

Select a ROI and then press SPACE or ENTER button!
Cancel the selection process by pressing c button!


Implementando Tracker para rastrear objetos

In [16]:
import math

class EuclideanDistTracker:
  def __init__(self):
    self.center_points = {}
    self.id_count = 0

  def update(self, objects_rect):
    objects_bbs_ids = []

    for rect in objects_rect:
      x, y, w, h = rect
      cx = (x + x + w) // 2
      cy = (y + y + h) // 2

      same_object_detected = False
      for id, pt in self.center_points.items():
        dist = math.hypot(cx - pt[0], cy - pt[1])

        if dist < 20:
          self.center_points[id] = (cx, cy)
          objects_bbs_ids.append([x, y, w, h, id])
          same_object_detected = True
          
      if not same_object_detected:
        self.center_points[self.id_count] = (cx, cy)
        objects_bbs_ids.append([x, y, w, h, self.id_count])
        self.id_count += 1

    return objects_bbs_ids

In [ ]:
# Criar o tracker
tracker = EuclideanDistTracker()

cap = cv2.VideoCapture(video_path)

_, first_frame = cap.read()

# Extrair região de interesse (ROI)
selected_roi = select_roi(first_frame)

while True:
  ret, frame = cap.read()

  if not ret:
    break

  frame = resize_frame(frame)

  # Extrair o ROI do frame atual
  roi = extract_roi(frame, selected_roi)

  # Detectar objetos em movimento dentro do ROI
  mask = object_detector.apply(roi)

  # Remover sombras
  _, mask = cv2.threshold(mask, 254, 255, cv2.THRESH_BINARY)

  contours, _ = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
  detections = []

  for contour in contours:
    # Calcular a área do contorno e remover pequenos contornos
    area = cv2.contourArea(contour)

    if area > 100:
      # cv2.drawContours(cur_roi, [contour], -1, (0, 255, 0), 2)
      x, y, w, h = cv2.boundingRect(contour)
      # cv2.rectangle(roi, (x, y), (x+w, y+h), (0, 255, 0), 2)

      detections.append([x, y, w, h])

  boxes_ids = tracker.update(detections)
  for box_id in boxes_ids:
    x, y, w, h, id = box_id
    cv2.putText(roi, str(id), (x, y - 10), cv2.FONT_HERSHEY_PLAIN, 1, (0, 0, 255), 1)
    cv2.rectangle(roi, (x, y), (x+w, y+h), (0, 255, 0), 2)

  cv2.imshow('frame', frame)
  cv2.imshow('mask', mask)

  key = cv2.waitKey(30)

  if key == ord('q'):
    break

cap.release()
cv2.destroyAllWindows()

Select a ROI and then press SPACE or ENTER button!
Cancel the selection process by pressing c button!


Refinamentos (TODO)

In [ ]:
# Criar o tracker
tracker = EuclideanDistTracker()

cap = cv2.VideoCapture(video_path)

_, first_frame = cap.read()

# Extrair região de interesse (ROI)
selected_roi = select_roi(first_frame)

while True:
  ret, frame = cap.read()

  if not ret:
    break

  frame = resize_frame(frame)

  # Extrair o ROI do frame atual
  roi = extract_roi(frame, selected_roi)

  # Detectar objetos em movimento dentro do ROI
  mask = object_detector.apply(roi)

  # Remover sombras
  _, mask = cv2.threshold(mask, 254, 255, cv2.THRESH_BINARY)

  contours, _ = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
  detections = []
  boxes = []

  for contour in contours:
    # Calcular a área do contorno e remover pequenos contornos
    area = cv2.contourArea(contour)

    if area > 100:
      # cv2.drawContours(cur_roi, [contour], -1, (0, 255, 0), 2)
      # cv2.rectangle(roi, (x, y), (x+w, y+h), (0, 255, 0), 2)

      contour_box = cv2.boundingRect(contour)
      centroid = (contour_box[0] + contour_box[2] // 2, contour_box[1] + contour_box[3] // 2)

      inside_another_rect = False

      for box in boxes:
        if cv2.pointPolygonTest(contour_box, centroid, False) >= 0:
          inside_another_rect = True
          break

      if not inside_another_rect:
        detections.append([x, y, w, h])

  boxes_ids = tracker.update(detections)
  for box_id in boxes_ids:
    x, y, w, h, id = box_id
    cv2.putText(roi, str(id), (x, y - 10), cv2.FONT_HERSHEY_PLAIN, 1, (0, 0, 255), 1)
    cv2.rectangle(roi, (x, y), (x+w, y+h), (0, 255, 0), 2)

  cv2.imshow('frame', frame)
  cv2.imshow('mask', mask)

  key = cv2.waitKey(0)

  if key == ord('q'):
    break

cap.release()
cv2.destroyAllWindows()

Select a ROI and then press SPACE or ENTER button!
Cancel the selection process by pressing c button!
